# Fit Models

In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.multiclass import OneVsRestClassifier

from tqdm_joblib import tqdm_joblib
from tqdm.auto import tqdm

from bioacoustics.data import load_results
from bioacoustics.metrics import evaluate_multilabel_model

from bioacoustics.modeling import FitMode, Classifier, HierarchicalMixtureOfExperts
from bioacoustics.modeling import split_data, get_prediction_pipeline

from bioacoustics.modeling import ignore_warnings

ignore_warnings()  # TODO: fix the reasons for these warnings

%load_ext autoreload
%autoreload 2

/Users/vlad/Documents/University/Master-MIND/bioacoustic-species-detection/.venv/lib/python3.13/site-packages/tqdm_joblib/__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


Problems to think about:

- Multi-label prediction -> hierarchical models? implement with `Pipeline`
- Class imbalance -> tinker with class weights, adjust probabilities?
- Use additional metadata from train, not available in soundscapes
- Maybe use hybrid approaches? learn about how to combine different models' predictions

think about chunking the audio, tuning the threshold for ech class

## Load features and labels

In [2]:
data_train = load_results("features", "data_train", frozen=True)
data_train_soundscapes = load_results("features", "data_train_soundscapes", frozen=True)

Note that it doesn't make much sense to validate on iNat or XC data, since it's of different format than test soundscapes.

In [3]:
FIT_MODE = FitMode.SOUNDSCAPE_TO_SOUNDSCAPE

X_train, X_test, y_class_train, y_class_test, y_primary_train, y_primary_test = (
    split_data(
        data_train,
        data_train_soundscapes,
        FIT_MODE,
        test_size=0.2,
        random_state=41,
    )
)

## Predict class

Start with predicting class only, we may after use this result when predicting primary label.



In [4]:
pipeline_class = get_prediction_pipeline(Classifier.XGBOOST)
with tqdm_joblib(desc="Training OvR", total=y_class_train.shape[1]):
    pipeline_class.fit(X_train, y_class_train)

Training OvR:   0%|          | 0/5 [00:00<?, ?it/s]

In [5]:
print(f"{FIT_MODE.name} predicting CLASS".center(60))
evaluate_multilabel_model(pipeline_class, X_test, y_class_test)

         SOUNDSCAPE_TO_SOUNDSCAPE predicting CLASS          

================== CLASSIFICATION REPORT ===================
              precision    recall  f1-score   support

    Amphibia       0.99      1.00      1.00       232
        Aves       0.76      0.68      0.72       112
     Insecta       1.00      1.00      1.00        72
    Mammalia       1.00      1.00      1.00         4
    Reptilia       1.00      0.25      0.40         8

   micro avg       0.94      0.90      0.92       428
   macro avg       0.95      0.79      0.82       428
weighted avg       0.93      0.90      0.91       428
 samples avg       0.96      0.94      0.93       428


================= THRESHOLD-BASED METRICS ==================
Macro F1:   0.8225
Micro F1:   0.9190
Hamming loss: 0.0439

============== RANKING & PROBABILITY METRICS ===============
Macro ROC AUC: 0.9739
Micro ROC AUC: 0.9824
Macro AP:      0.9508
Micro AP:      0.9728
LRAP:          0.9919


## Predict species (primary label)

We can use these approaches:

1. **Hierarchical classification:**
    - first predict the class and then train per-class model
    - we must avoid training mismatch - at inference predicted class is used, but at training use only true class, so we better condition on predicted class
    - problems:
        - error propagation if predicted class is wrong
    - **soft version:**
        - $P(s|x) = \sum_c P(s|x,c)P(c|x)$
        - predict class probabilities and use them as soft weights for species classifiers
        - i.e. mixture of experts, soft routing
        - pros:
            - avoids routing errors
            - more consistent as a probability model
    - ideas:
        - I'm afraid class signal will be hidden by hundreds of features, maybe we should emphasize it?
    - multilabel problem:
        - since data is multilabel, classes are not mutually exclusive, it means that when we do an averages sum of preicted probabilities, it will not sum to 1
        - so we would need to normalize it or just do average prediction across experts

1. **Feature augmentation**
    - include predicted class as a new feature:
    - no hard routing, so more robust
    - simple to implement
    - problems:
        - this feature can be noisy
    - **soft version:**
        - include predicted class probabilities as new features

1. Other ideas:
    - hierarchical loss (penalize wrong species but good class less)

We would focus on soft approaches as they are more robust since they avoid noisy hard class assignment.

In [6]:
# TODO: think about OOF prediction (for training data, otherwise prediction is too optimistic)
y_class_train_proba = pipeline_class.predict_proba(X_train)
y_class_test_proba = pipeline_class.predict_proba(X_test)

### Baseline (ignore predicted class)

In [7]:
pipeline_baseline = get_prediction_pipeline(Classifier.XGBOOST)
with tqdm_joblib(desc="Training OvR", total=y_primary_train.shape[1]):
    pipeline_baseline.fit(X_train, y_primary_train)

Training OvR:   0%|          | 0/234 [00:00<?, ?it/s]

In [8]:
print(f"{FIT_MODE.name} predicting SPECIES".center(60))
print("(ignore CLASS)".center(60))
evaluate_multilabel_model(pipeline_baseline, X_test, y_primary_test)

        SOUNDSCAPE_TO_SOUNDSCAPE predicting SPECIES         
                       (ignore CLASS)                       

================= THRESHOLD-BASED METRICS ==================
Macro F1:   0.0455
Micro F1:   0.5484
Hamming loss: 0.0131

============== RANKING & PROBABILITY METRICS ===============
Macro ROC AUC: nan
Micro ROC AUC: 0.9457
Macro AP:      0.0681
Micro AP:      0.5920
LRAP:          0.6345


### Soft hierarchical classification

In [9]:
expert_pipelines = HierarchicalMixtureOfExperts(y_class_test.shape[1], Classifier.XGBOOST)
expert_pipelines.fit(X_train, y_class_train, y_primary_train)

Training expert for Amphibia  :   0%|          | 0/234 [00:00<?, ?it/s]

Training expert for Aves      :   0%|          | 0/234 [00:00<?, ?it/s]

Training expert for Insecta   :   0%|          | 0/234 [00:00<?, ?it/s]

Training expert for Mammalia  :   0%|          | 0/234 [00:00<?, ?it/s]

Training expert for Reptilia  :   0%|          | 0/234 [00:00<?, ?it/s]

In [10]:
print(f"{FIT_MODE.name} predicting SPECIES".center(60))
print("(soft mixture of CLASS experts)".center(60))
evaluate_multilabel_model(expert_pipelines, X_test, y_primary_test, y_class_test_proba)

        SOUNDSCAPE_TO_SOUNDSCAPE predicting SPECIES         
              (soft mixture of CLASS experts)               

================= THRESHOLD-BASED METRICS ==================
Macro F1:   0.0528
Micro F1:   0.5858
Hamming loss: 0.0129

============== RANKING & PROBABILITY METRICS ===============
Macro ROC AUC: nan
Micro ROC AUC: 0.9479
Macro AP:      0.0760
Micro AP:      0.5533
LRAP:          0.6454


### Soft feature augmentation

In [11]:
X_test_augmented = np.concatenate([X_test, y_class_test_proba], axis=1)
X_train_augmented = np.concatenate([X_train, y_class_train_proba], axis=1)

pipeline_augmentation = get_prediction_pipeline(Classifier.XGBOOST)

with tqdm_joblib(desc="Training OvR", total=y_primary_train.shape[1]):
    pipeline_augmentation.fit(X_train_augmented, y_primary_train)

Training OvR:   0%|          | 0/234 [00:00<?, ?it/s]

In [12]:
print(f"{FIT_MODE.name} predicting SPECIES".center(60))
print("(CLASS feature augmentation)".center(60))
evaluate_multilabel_model(pipeline_augmentation, X_test_augmented, y_primary_test)

        SOUNDSCAPE_TO_SOUNDSCAPE predicting SPECIES         
                (CLASS feature augmentation)                

================= THRESHOLD-BASED METRICS ==================
Macro F1:   0.0424
Micro F1:   0.5586
Hamming loss: 0.0128

============== RANKING & PROBABILITY METRICS ===============
Macro ROC AUC: nan
Micro ROC AUC: 0.9450
Macro AP:      0.0698
Micro AP:      0.5907
LRAP:          0.6441


## Postprocessing

In [ ]:
# TODO: do temporal smoothing of predicted probabilities (since we know the file and the timestamp)

## Interpret the model

In [13]:
# TODO: get feature importance

# SANDBOX

In [14]:
# TODO: try XGBoost, RF etc, change multi-label strategies
# TODO: hierarchical approach - train a model per class
# and predict species conditioned on class (can try soft versions - multiply probas)

# TODO: alternative to hierarchical approach - include predicted class as a feature
# for species prediction
# TODO: the same idea for training metadata that is absent for the test
# - we can learn to predict this metadata as a secondary task and then include
# it in the model
# NOTE: attention to data leak between train - validation when using secondary tasks
# to generate features (class, metadata)

# TODO: can use this metadata as well for stratification - better split validation
# (make sure that there is no data leak because of the same site)
# TODO: check whether metadata alone can predict species (check for shortcut learning)

# TODO: smart cross validations

# TODO: quality of train audio from iNat and xeno-cant is poorer and further from test than train soundscapes,
#       so maybe we should use them only for species poorly covered by soundscapes

# TODO: impute some NaNs

# TODO: maybe stratify be species or sth else since train data had
#   too (really) many bird recordings, whereas soundscapes contain more amphibians

# TODO: maybe play with thresholds

# TODO: temporal smoothing of predicted probabilities!